In [1]:
import numpy as np
import pandas as pd
import re
import unicodedata
import glob
import os

# Process all raw data

In [9]:
read_base_path = "receipt_raw"
save_base_path = "receipt"
folder = ['0', '1', '2']

colums_no_need = ['載具自訂名稱', '發票號碼', '發票金額', '發票狀態', '折讓', '賣方統一編號', '賣方地址', '買方統編', ]

## Clean store & item name


*   全形轉半形
*   全部轉小寫
*   去除前後空白、重複空白只留一個
*   保留英文、中文、空白、-/()+&%_
*   針對 store name 移除「股份有限公司/分公司」等字眼
*   對空白的 store or item name 給予 `[UNK_STORE]`, `[UNK_ITEM]` 的標籤

In [3]:
def fullwidth_to_halfwidth(text):
    if pd.isna(text):
        return text

    result = ""

    for char in str(text):
        code = ord(char)

        # 全形空白
        if code == 0x3000:
            result += " "

        # 全形字元
        elif 0xFF01 <= code <= 0xFF5E:
            result += chr(code - 0xFEE0)

        else:
            result += char

    return result

In [4]:
REMOVE_STORE_WORDS = [
    "股份有限公司",
    "有限公司",
    "分公司",
    "股份有限",
    "corp",
    "ltd"
]

In [5]:
def clean_name(text_type, text):

    null_tag = ""
    if text_type == "store":
        null_tag = "[UNK_STORE]"
    else:
        null_tag = "[UNK_ITEM]"

    if pd.isna(text) or str(text).strip() == "":
        return null_tag

    text = str(text)

    # 全形轉半形
    text = fullwidth_to_halfwidth(text)

    # 小寫
    text = text.lower()

    # 去前後空白
    text = text.strip()

    # 多空白變單一空白
    text = re.sub(r"\s+", " ", text)

    # 只移除非常奇怪的符號
    text = re.sub(r"[^\w\u4e00-\u9fff\s\-/()+&%]", "", text)

    # 移除公司字樣
    if text_type == "store":
        for word in REMOVE_STORE_WORDS:
            text = text.replace(word, "")

    if text == "":
        return null_tag

    return text

In [10]:
for i in range(len(folder)):
    read_path = f"{read_base_path}/{folder[i]}"
    save_path = f"{save_base_path}/{folder[i]}"

    os.makedirs(save_path, exist_ok=True)

    files = glob.glob(f'{read_path}/*.csv')

    for f in files:
        df = pd.read_csv(
            f,
            usecols=range(14),
            engine='python'
        )

        df.drop(columns=colums_no_need, inplace=True)
        df.dropna(subset=['消費明細_金額'], inplace=True)

        df['store_clean'] = df['賣方名稱'].apply(lambda x: clean_name("store", x))
        df['item_clean'] = df['消費明細_品名'].apply(lambda x: clean_name("item", x))

        filename = os.path.basename(f)
        print("processing:", filename)

        df.to_csv(f'{save_path}/{filename}', index=False)

processing: 202509.csv
processing: 202605.csv
processing: 202511.csv
processing: 202603.csv
processing: 202512.csv
processing: 202602.csv
processing: 202510.csv
processing: 202601.csv
processing: 202604.csv
processing: 202603_1.csv
processing: 202509_1.csv
processing: 202512_1.csv
processing: 202509_2.csv
processing: 202603_2.csv
processing: 202511_1.csv
processing: 202601_2.csv
processing: 202601_1.csv
processing: 202604_2.csv
processing: 202511_2.csv
processing: 20260501_0526.csv
processing: 202602.csv
processing: 202510.csv
processing: 202604_1.csv
processing: 202512_2.csv
processing: 20251101-1130.csv
processing: 20260301-0331.csv
processing: 20251001-1031.csv
processing: 20260201-0228.csv
processing: 20251201-1231.csv
processing: 20260101-0131.csv
processing: 20260401-0430.csv
processing: 20260501-0525.csv


## Concat all data

In [11]:
for name in folder:
    save_path = f"{save_base_path}/{name}"

    df_list = []

    files = glob.glob(f'{save_path}/*.csv')

    for f in files:
        df_list.append(pd.read_csv(f, engine='python'))

    if len(df_list) == 0:
        print(f"{name} is empty, skip")
        continue

    df = pd.concat(df_list, ignore_index=True)

    # 把日期轉成 datetime
    df["發票日期"] = pd.to_datetime(
        df["發票日期"].astype(str).str.replace(".0", "", regex=False), # 把原本的 float 改成 string
        format="%Y%m%d",
        errors="coerce"
    )
    # 依照時間排序
    df.sort_values(by="發票日期", inplace=True)

    os.makedirs(f"{save_path}/all", exist_ok=True)
    df.to_csv(f"{save_path}/all/all.csv", index=False)

    df_clean = df.drop(columns=["賣方名稱", "消費明細_品名"])
    df_clean.to_csv(f"{save_path}/all/all_clean.csv", index=False)

    print("processing:", save_path)

processing: receipt/0
processing: receipt/1
processing: receipt/2


# 處理 LLM 預標注結果

In [15]:
path = "receipt"

## 移除折扣消費（消費金額 < 0）

In [19]:
llm_columns = ["confidence", "reason", "llm_origin"]

os.makedirs(f'{path}/labeled', exist_ok=True)

files = glob.glob(f'{path}/labeled/*.csv')

for f in files:
    df = pd.read_csv(f)
    df = df.drop(columns=llm_columns)

    # 移除一些折扣的消費
    df = df[df["消費明細_金額"] > 0]

    filename = os.path.basename(f)
    print("processing:", filename)

    # 統計一個 user 的資料分布
    print(df["label"].value_counts())
    print(df["label"].value_counts(normalize=True))

    os.makedirs(f"{path}/train", exist_ok=True)
    df.to_csv(f'{path}/train/{filename}', index=False)

processing: all_clean_labeled_1.csv
label
飲食      588
購物       38
教育       35
交通        9
醫療健康      7
娛樂        2
3C電子      1
Name: count, dtype: int64
label
飲食      0.864706
購物      0.055882
教育      0.051471
交通      0.013235
醫療健康    0.010294
娛樂      0.002941
3C電子    0.001471
Name: proportion, dtype: float64
processing: all_clean_labeled_2.csv
label
飲食      1241
交通       155
娛樂        68
購物        27
教育         6
醫療健康       5
3C電子       3
Name: count, dtype: int64
label
飲食      0.824585
交通      0.102990
娛樂      0.045183
購物      0.017940
教育      0.003987
醫療健康    0.003322
3C電子    0.001993
Name: proportion, dtype: float64
processing: all_clean_labeled_3.csv
label
飲食      324
購物       60
教育       28
醫療健康     12
3C電子      2
娛樂        2
Name: count, dtype: int64
label
飲食      0.757009
購物      0.140187
教育      0.065421
醫療健康    0.028037
3C電子    0.004673
娛樂      0.004673
Name: proportion, dtype: float64


In [20]:
os.makedirs(path, exist_ok=True)

files = glob.glob(f'{path}/train/*.csv')
user_id = 0
df_list = []

for f in files:
    df = pd.read_csv(f)
    df["user_id"] = user_id

    filename = os.path.basename(f)
    print("processing:", filename)

    df_list.append(df)
    df_concat = pd.concat(df_list, ignore_index=True)

    user_id += 1

os.makedirs(f"{path}/train", exist_ok=True)
df_concat.to_csv(f'{path}/train/all_user.csv', index=False)

processing: all_clean_labeled_1.csv
processing: all_clean_labeled_2.csv
processing: all_clean_labeled_3.csv


In [21]:
all_user_df = pd.read_csv(f'{path}/train/all_user.csv')

In [22]:
all_user_df["label"].value_counts()

,count
label,
飲食,2153
交通,164
購物,125
娛樂,72
教育,69
醫療健康,24
3C電子,6


In [23]:
all_user_df["label"].value_counts(normalize=True)

,proportion
label,
飲食,0.823957
交通,0.062763
購物,0.047838
娛樂,0.027555
教育,0.026406
醫療健康,0.009185
3C電子,0.002296


## 合併過少筆數的類別：最後分成 6 大類

In [24]:
all_user_df["label"] = all_user_df["label"].replace({
    "3C電子": "購物"
})

## 對 label 設定 label_id

In [25]:
label2id = {
    "飲食": 0,
    "交通": 1,
    "購物": 2,
    "娛樂": 3,
    "教育": 4,
    "醫療健康": 5,
}

id2label = {v:k for k,v in label2id.items()}


all_user_df["label_id"] = all_user_df["label"].map(label2id)

## 為消費金額設立消費區間

In [26]:
def price_bucket(x):

    if x < 50:
        return "VERY_LOW"

    elif x < 150:
        return "LOW"

    elif x < 500:
        return "MID"

    elif x < 1500:
        return "HIGH"

    else:
        return "VERY_HIGH"

In [27]:
all_user_df["price_bucket"] = all_user_df["消費明細_金額"].apply(price_bucket)

In [30]:
all_user_df.to_csv("all_user_6_label.csv")

In [29]:
all_user_df

,發票日期,消費明細_數量,消費明細_單價,消費明細_金額,store_clean,item_clean,label,user_id,label_id,price_bucket
0,2025-09-01,1.0,115.0,115.0,福康事業,法式特餐,飲食,0,0,LOW
1,2025-09-01,1.0,95.0,95.0,一之軒食品南西分店,生吐司,飲食,0,0,LOW
2,2025-09-02,1.0,990.0,990.0,新光三越百貨台北信義,咖啡廳coffee shops & tea salons,飲食,0,0,HIGH
3,2025-09-02,1.0,80.0,80.0,新光三越百貨台北信義,小吃food court,飲食,0,0,LOW
4,2025-09-03,1.0,35.0,35.0,統一超商台北市第148,桂格100%燕麥(顆粒微甜)290ml,飲食,0,0,VERY_LOW
...,...,...,...,...,...,...,...,...,...,...
2608,2026-05-21,1.0,39.0,39.0,全家便利商店台大二活門市部,香蕉可可三明治,飲食,2,0,VERY_LOW
2609,2026-05-21,1.0,38.0,38.0,全家便利商店新北市第一一二,午后時光重乳奶茶,飲食,2,0,VERY_LOW
2610,2026-05-22,1.0,65.0,65.0,和德昌台中學士路,雙倍or冰炫風,飲食,2,0,LOW
2611,2026-05-23,1.0,330.0,330.0,勤美台中,星球工坊爆米花,飲食,2,0,MID
